# Ejercicio 1: tres formas de validar el mismo modelo

`viajes_features.csv` y la preparación importada de `preparacion_taxis.py`, como en las Semanas 10 y 11. Hasta ahora, cada modelo se midió una vez, sobre los mismos 595 viajes de prueba. Hoy se mide con validación cruzada, con tres esquemas distintos, y con más de una métrica. `evaluar` recibe el esquema y guarda en `resultados` el error de cada pliegue, su promedio y su desviación.

Corre todas las celdas antes de tocar nada (**Run All**): corre de principio a fin sin los `TODO` resueltos.

In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from preparacion_taxis import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
COLUMNAS = NUMERICAS + YA_LISTAS + CATEGORICAS
X = viajes[COLUMNAS]
y = viajes[OBJETIVO]

# Los mismos viajes, ordenados del mas antiguo al mas reciente.
viajes_en_orden = viajes.sort_values("fecha_hora_inicio").reset_index(drop=True)
X_en_orden = viajes_en_orden[COLUMNAS]
y_en_orden = viajes_en_orden[OBJETIVO]
print("Viajes:", len(viajes), " desde", viajes_en_orden["fecha_hora_inicio"].iloc[0], "hasta", viajes_en_orden["fecha_hora_inicio"].iloc[-1])

Viajes: 2975  desde 2026-09-01 00:14 hasta 2026-10-31 23:46


## La función que valida cada modelo

In [2]:
resultados = {}


def gradient_boosting():
    return GradientBoostingRegressor(random_state=42)


def bosque():
    return RandomForestRegressor(n_estimators=200, random_state=42)


def evaluar(nombre, modelo, esquema, X, y, grupos=None):
    """Validacion cruzada con el esquema dado. Guarda en resultados el error
    absoluto medio de cada pliegue, su promedio y su desviacion."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, groups=grupos, scoring="neg_mean_absolute_error")
    errores = -puntajes["test_score"]
    resultados[nombre] = {
        "validacion": nombre,
        "mae_promedio": round(errores.mean(), 4),
        "desviacion": round(errores.std(), 4),
        "pliegues": [round(error, 2) for error in errores],
    }
    print(f"{nombre:28s} promedio {errores.mean():.4f}  desviacion {errores.std():.4f}  pliegues {resultados[nombre]['pliegues']}")

## Lo que dijo la Semana 10: una sola división

In [3]:
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
una_division = {}
for nombre, modelo in [("gradient boosting", gradient_boosting()), ("bosque aleatorio", bosque())]:
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)]).fit(X_ent, y_ent)
    una_division[nombre] = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    print(f"Una division, {nombre}: {una_division[nombre]:.4f}")

Una division, gradient boosting: 1.2754
Una division, bosque aleatorio: 1.2989


## Paso 1: validación cruzada al azar

### Baseline antes del TODO

In [4]:
# [DONE] 1: validacion cruzada al azar, de 5 pliegues, mezclando los viajes
# antes de repartirlos, con random_state=42.

#? Si queremos mezclar los viajes antes de repartilos, podemos usar KFold con shuffle=True y random_state=42.
esquema_aleatorio = KFold(n_splits=2, shuffle=False)

evaluar("aleatorio, gradient boosting", gradient_boosting(), esquema_aleatorio, X, y)
evaluar("aleatorio, bosque", bosque(), esquema_aleatorio, X, y)

aleatorio, gradient boosting promedio 1.3015  desviacion 0.0555  pliegues [np.float64(1.25), np.float64(1.36)]
aleatorio, bosque            promedio 1.3932  desviacion 0.0680  pliegues [np.float64(1.33), np.float64(1.46)]


In [5]:
# [DONE] 1: validacion cruzada al azar, de 5 pliegues, mezclando los viajes
# antes de repartirlos, con random_state=42.

#? Si queremos mezclar los viajes antes de repartilos, podemos usar KFold con shuffle=True y random_state=42.
esquema_aleatorio = KFold(n_splits=5, shuffle=True, random_state=42)

evaluar("aleatorio, gradient boosting", gradient_boosting(), esquema_aleatorio, X, y)
evaluar("aleatorio, bosque", bosque(), esquema_aleatorio, X, y)

aleatorio, gradient boosting promedio 1.2288  desviacion 0.0361  pliegues [np.float64(1.28), np.float64(1.24), np.float64(1.16), np.float64(1.24), np.float64(1.23)]
aleatorio, bosque            promedio 1.2887  desviacion 0.0286  pliegues [np.float64(1.3), np.float64(1.27), np.float64(1.25), np.float64(1.33), np.float64(1.29)]


## Paso 2: validación temporal

In [6]:
# [DONE] 2: validacion temporal de 5 pliegues: cada pliegue entrena con los
# viajes anteriores y valida con los siguientes. Usa los viajes en orden.
esquema_temporal = TimeSeriesSplit(n_splits=5) #? Debido a que no se define ninguna modificacion adicional, no se configura el mecanismo de splitting con  el tamanio del test, etc.

evaluar("temporal, gradient boosting", gradient_boosting(), esquema_temporal, X_en_orden, y_en_orden)

temporal, gradient boosting  promedio 1.3099  desviacion 0.0693  pliegues [np.float64(1.4), np.float64(1.21), np.float64(1.38), np.float64(1.27), np.float64(1.29)]


## Paso 3: validación por grupo

In [8]:
# TODO 3: validacion por grupo de 5 pliegues, con la zona de origen como
# grupo: ninguna zona aparece a la vez en entrenamiento y en validacion.
esquema_por_grupo = GroupKFold(n_splits=5)
grupos = X["zona_origen"]

evaluar("por zona, gradient boosting", gradient_boosting(), esquema_por_grupo, X, y, grupos)

por zona, gradient boosting  promedio 2.2996  desviacion 2.0648  pliegues [np.float64(1.27), np.float64(1.31), np.float64(1.29), np.float64(1.19), np.float64(6.43)]


## Paso 4: la métrica según el costo del error

In [11]:
modelo = Pipeline([("preparar", preparacion()), ("modelo", gradient_boosting())]).fit(X_ent, y_ent)
estimada = modelo.predict(X_prueba)
real = y_prueba.to_numpy()

mae = mean_absolute_error(real, estimada)
# [DONE] 4: la raiz del error cuadratico medio (rmse), y la proporcion de
# viajes de prueba en que la tarifa real supera a la estimada en mas de
# 20% (subestimados_20).
rmse = root_mean_squared_error(real, estimada)
subestimados_20 = (np.sum(real > estimada * 1.2) / len(real)) * 100

print(f"MAE {mae:.4f}   RMSE {rmse}   viajes subestimados en mas de 20%: {subestimados_20}")

MAE 1.2754   RMSE 1.9213018246369917   viajes subestimados en mas de 20%: 4.369747899159663


## Tabla de resultados

In [12]:
tabla = pd.DataFrame(resultados.values())
print(tabla.to_string(index=False))

                  validacion  mae_promedio  desviacion                       pliegues
aleatorio, gradient boosting        1.2288      0.0361 [1.28, 1.24, 1.16, 1.24, 1.23]
           aleatorio, bosque        1.2887      0.0286  [1.3, 1.27, 1.25, 1.33, 1.29]
 temporal, gradient boosting        1.3099      0.0693  [1.4, 1.21, 1.38, 1.27, 1.29]
 por zona, gradient boosting        2.2996      2.0648 [1.27, 1.31, 1.29, 1.19, 6.43]
